# 031 — Evaluation (heteroscedastic / NLL)

Quantitative and qualitative evaluation of `attention_unet_nll` (trained in `021_training_nll.ipynb`) on the held-out test set.

Mirrors `030_evaluation.ipynb`, with two differences that follow directly from the two-channel `(mu, log_var)` output: metrics are computed from the `mu` channel only (`scripts.metrics.Mu*Metric`), and a new closing section visualises the model's own learned anomaly z-score `(real_IR - mu) / sigma` — see `code-review.md` §7.6.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    grouped_train_val_test_split,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.trainer_nll import load_model_nll
from scripts.visualization import plot_predictions
from scripts.visualization_nll import plot_predictions_nll, plot_zscore

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Build test dataset

In [ ]:
# Artworks split logic: every artwork ID (including the paint-on-support
# mockup groups) is a group kept entirely within a single fold, to prevent
# leakage between sections of the same painting. Must match the split used
# for training (021_training_nll.ipynb) so the test fold reported here is
# disjoint from what the loaded checkpoint was trained/validated on.
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
_, _, test_pairs = grouped_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    seed=settings.SEED,
)

test_ds = build_dataset(
    test_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)
print(f"Test pairs: {len(test_pairs)} | Test batches: {len(test_ds)}")

In [ ]:
# Artwork and mockups split logic (active): real artworks are still
# grouped and leakage-free, but the mockup groups in
# settings.MOCKUP_ARTWORK_IDS only contribute a small fraction of their
# sections to test (default 5%), the rest going to train/val. Overwrites
# test_pairs from the block above — only use this if the checkpoint being
# evaluated was itself trained with mockup_aware_train_val_test_split
# (021_training_nll.ipynb) — the two split functions do not produce the
# same test fold, so mixing them would leak train data into the reported
# test metrics.
pairs_mockups = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
_, _, test_pairs = mockup_aware_train_val_test_split(
    pairs_mockups,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    seed=settings.SEED,
)

test_ds = build_dataset(
    test_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)
print(f"Test pairs: {len(test_pairs)} | Test batches: {len(test_ds)}")

## 2. Evaluate model

In [ ]:
ARCHS = ["attention_unet_nll"]
results: dict = {}

for arch in ARCHS:
    try:
        model = load_model_nll(arch, model_dir=settings.MODELS_DIR)
    except FileNotFoundError as exc:
        print(f"[skip] {exc}")
        continue

    print(f"Evaluating {arch}...")
    metrics = model.evaluate(test_ds, verbose=0, return_dict=True)
    results[arch] = metrics
    print(f"  {arch}: { {k: f'{v:.4f}' for k, v in metrics.items()} }")

## 3. Comparison table

`loss` here is the Gaussian NLL, not `combined_loss` — not comparable in scale to `030_evaluation.ipynb`'s loss column. `mae`/`ssim`/`psnr` are comparable, since they are computed the same way from `mu` in both notebooks.

In [ ]:
if results:
    col_w = 14
    headers = ["architecture"] + list(next(iter(results.values())).keys())
    print("".join(h.ljust(col_w) for h in headers))
    print("-" * (col_w * len(headers)))
    for arch, metrics in results.items():
        row = [arch] + [f"{v:.4f}" for v in metrics.values()]
        print("".join(c.ljust(col_w) for c in row))

## 4. Bar chart comparison

In [ ]:
if results:
    metric_keys = list(next(iter(results.values())).keys())
    n = len(metric_keys)
    archs = list(results.keys())
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle("Architecture comparison — test set")
    for ax, key in zip(axes, metric_keys):
        vals = [results[a][key] for a in archs]
        bars = ax.bar(archs, vals)
        ax.set_title(key.upper())
        ax.set_xticklabels(archs, rotation=20, ha="right")
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f"{v:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    plt.tight_layout()
    plt.show()

## 5. Qualitative gallery — 5 test samples

`sigma = exp(0.5 * log_var)`.

In [ ]:
N_DISPLAY = 5

for arch in ARCHS:
    try:
        model = load_model_nll(arch, model_dir=settings.MODELS_DIR)
    except FileNotFoundError:
        continue

    print(f"\n--- {arch} ---")
    sample_ds = build_dataset(test_pairs[:N_DISPLAY], batch_size=1, augment=False, shuffle=False)

    for (rgb_batch, ir_batch), pair in zip(sample_ds, test_pairs[:N_DISPLAY]):
        pred = model.predict(rgb_batch, verbose=0)[0]
        mu, log_var = pred[..., 0:1], pred[..., 1:2]
        sigma = np.exp(0.5 * log_var)
        fig = plot_predictions_nll(
            rgb_batch[0].numpy(),
            ir_batch[0].numpy(),
            mu,
            sigma,
            title=f"{arch} — {Path(pair[0]).stem}",
        )
        plt.show()

## 6. Worst predictions (lowest SSIM on `mu`)

In [ ]:
N_WORST = 3

for arch in ARCHS:
    try:
        model = load_model_nll(arch, model_dir=settings.MODELS_DIR)
    except FileNotFoundError:
        continue

    ssim_scores: list[tuple[float, int]] = []
    single_ds = build_dataset(test_pairs, batch_size=1, augment=False, shuffle=False)

    for idx, (rgb_batch, ir_batch) in enumerate(single_ds):
        pred = model.predict(rgb_batch, verbose=0)
        mu = pred[..., 0:1]
        ssim_val = float(
            tf.reduce_mean(tf.image.ssim(ir_batch, tf.constant(mu), max_val=1.0))
        )
        ssim_scores.append((ssim_val, idx))

    worst = sorted(ssim_scores)[:N_WORST]
    print(f"\n--- {arch}: {N_WORST} worst predictions (SSIM) ---")
    for ssim_val, idx in worst:
        rgb_path, ir_path = test_pairs[idx]
        rgb_batch, ir_batch = next(iter(build_dataset([test_pairs[idx]], batch_size=1)))
        pred = model.predict(rgb_batch, verbose=0)[0]
        mu, log_var = pred[..., 0:1], pred[..., 1:2]
        sigma = np.exp(0.5 * log_var)
        fig = plot_predictions_nll(
            rgb_batch[0].numpy(),
            ir_batch[0].numpy(),
            mu,
            sigma,
            title=f"{arch} | SSIM={ssim_val:.3f} | {Path(rgb_path).stem}",
        )
        plt.show()

## 7. Learned anomaly z-score

New section, specific to the heteroscedastic model: `z = (real_IR - mu) / sigma`, the learned counterpart of the fixed Gaussian-window normalisation used in `delta_analysis.py`. A pixel with a large raw delta but high learned `sigma` (a color/context known to vary a lot in training) yields a small, unremarkable `z`; a pixel with a modest raw delta but low `sigma` yields a large `z` — a genuine anomaly candidate. See `code-review.md` §7.6.

In [ ]:
for arch in ARCHS:
    try:
        model = load_model_nll(arch, model_dir=settings.MODELS_DIR)
    except FileNotFoundError:
        continue

    for (rgb_batch, ir_batch), pair in zip(
        build_dataset(test_pairs[:N_DISPLAY], batch_size=1, augment=False, shuffle=False),
        test_pairs[:N_DISPLAY],
    ):
        pred = model.predict(rgb_batch, verbose=0)[0]
        mu, log_var = pred[..., 0:1], pred[..., 1:2]
        sigma = np.exp(0.5 * log_var)
        fig = plot_zscore(
            ir_batch[0].numpy(),
            mu,
            sigma,
            title=f"{arch} — {Path(pair[0]).stem}",
        )
        plt.show()